# ABO150 QLoRA For Qwen3 (Colab)

Этот ноутбук запускает полный цикл для `dataset/abo_150_expanded`:
1. ставит зависимости;
2. клонирует публичный репозиторий;
3. подхватывает `HF_TOKEN` и `HF_USERNAME` из Colab Secrets;
4. собирает `train/val` dataset для QLoRA;
5. дообучает `qwen3_vl_8b`;
6. при желании пушит adapter на Hugging Face Hub;
7. валидирует adapter на фиксированном holdout `50` объектов.

Colab Secrets:
- `HF_TOKEN`
- `HF_USERNAME` (опционально, но удобно для автогенерации repo id)
- `comet_api_key` / `comet_workspace` / `comet_project_name` — опционально


In [ ]:
# Dependencies (Colab-safe versions + one-time auto-restart)
import os
import sys
import subprocess
from pathlib import Path

MARKER = Path('/tmp/abo150_qlora_qwen3_deps_ready')

if not MARKER.exists():
    print('Installing dependencies (first run)...')
    install_cmds = [
        [sys.executable, '-m', 'pip', 'uninstall', '-y', 'numpy'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--force-reinstall', 'numpy==2.1.3'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'transformers>=4.49.0,<5.0.0', 'accelerate', 'bitsandbytes', 'peft', 'sentencepiece', 'huggingface_hub'],
        [sys.executable, '-m', 'pip', 'install', '--no-cache-dir', '--upgrade', '--upgrade-strategy', 'only-if-needed',
         'pandas==2.2.2', 'pillow<12', 'pyyaml', 'boto3', 'rembg', 'onnxruntime', 'comet_ml'],
    ]

    for cmd in install_cmds:
        print('>>', ' '.join(cmd))
        subprocess.run(cmd, check=True)

    MARKER.write_text('ok')
    print('Dependencies installed. Restarting runtime now...')
    os.kill(os.getpid(), 9)
else:
    print('Dependencies already installed in this runtime. Continue.')


In [ ]:
# Clone or update public repo in Colab
import subprocess
from pathlib import Path

REPO_URL = 'https://github.com/Yaitco/VLM-2D-Physics-Boundaries.git'
WORKDIR = Path('/content/VLM-2D-Physics-Boundaries')

if not WORKDIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(WORKDIR)], check=True)
else:
    print(f'Repo already exists: {WORKDIR}. Pulling latest...')
    subprocess.run(['git', '-C', str(WORKDIR), 'pull', '--ff-only'], check=True)

%cd /content/VLM-2D-Physics-Boundaries


In [ ]:
# Load secrets and export env vars
import os

def _get_secret(name: str):
    try:
        from google.colab import userdata
        value = userdata.get(name)
        if value:
            return value
    except Exception:
        pass
    return os.getenv(name)

HF_TOKEN = _get_secret('HF_TOKEN')
HF_USERNAME = _get_secret('HF_USERNAME')

if HF_TOKEN:
    os.environ['HF_TOKEN'] = HF_TOKEN
if HF_USERNAME:
    os.environ['HF_USERNAME'] = HF_USERNAME

try:
    from huggingface_hub import login
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=True)
        print('HF token configured.')
    else:
        print('HF_TOKEN is missing. You can still build dataset, but push-to-hub will fail.')
except Exception as exc:
    print('HF login skipped:', exc)

print('HF_USERNAME:', HF_USERNAME or '<missing>')


In [ ]:
# QLoRA experiment config
from pathlib import Path

DATASET_NAME = 'abo_150_expanded'
PROTOCOL_NAME = 'full_expanded'
BASE_MODEL_KEY = 'qwen3_vl_8b'

TRAIN_IDS_PATH = Path('dataset/abo_150_expanded/splits/seed42_val50_train100/train_ids.txt')
VAL_IDS_PATH = Path('dataset/abo_150_expanded/splits/seed42_val50_train100/val_ids.txt')

PROPERTY_KEYS = ['intrinsic.main_material']
OUTPUT_TAG = 'abo150_qwen3_main_material'
TRAINSET_DIR = Path('outputs') / f'{OUTPUT_TAG}_dataset'
ADAPTER_DIR = Path('outputs') / OUTPUT_TAG

PUSH_TO_HUB = True
HUB_MODEL_ID = None  # e.g. 'your-hf-name/abo150-qwen3-main-material'
HUB_PRIVATE = False

NUM_TRAIN_EPOCHS = 6
LEARNING_RATE = 2e-4
GRAD_ACC_STEPS = 8
TRAIN_BATCH_SIZE = 1
EVAL_BATCH_SIZE = 1
LOGGING_STEPS = 5
SAVE_STEPS = 25
EVAL_STEPS = 25
DISABLE_TQDM = False


VALIDATION_VARIANTS = ['raw']

COMET_ENABLED = True
COMET_PROJECT_NAME = 'vlm-physics-training'
CUSTOM_MODEL_KEY = f'{BASE_MODEL_KEY}_qlora_{OUTPUT_TAG}'

print('Base model:', BASE_MODEL_KEY)
print('Property keys:', PROPERTY_KEYS)
print('Train ids:', TRAIN_IDS_PATH)
print('Val ids:', VAL_IDS_PATH)
print('Trainset dir:', TRAINSET_DIR)
print('Adapter dir:', ADAPTER_DIR)
print('Push to hub:', PUSH_TO_HUB)
if HUB_MODEL_ID:
    print('Hub model id:', HUB_MODEL_ID)
elif os.getenv('HF_USERNAME'):
    print('Hub model id (auto):', f"{os.getenv('HF_USERNAME')}/{ADAPTER_DIR.name}")
else:
    print('Hub model id: will be inferred from token if possible')


In [ ]:
# Sanity-check split files
import json

train_ids = [line.strip() for line in TRAIN_IDS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
val_ids = [line.strip() for line in VAL_IDS_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]

print('Train ids:', len(train_ids))
print('Val ids:', len(val_ids))
print('First train ids:', train_ids[:5])
print('First val ids:', val_ids[:5])


In [ ]:
# Build per-property QLoRA dataset
import subprocess

cmd = [
    'python', 'scripts/build_abo150_qlora_dataset.py',
    '--protocol-name', PROTOCOL_NAME,
    '--property-keys', ','.join(PROPERTY_KEYS),
    '--train-ids-path', str(TRAIN_IDS_PATH),
    '--val-ids-path', str(VAL_IDS_PATH),
    '--output-dir', str(TRAINSET_DIR),
]
print('>>', ' '.join(cmd))
subprocess.run(cmd, check=True)

manifest = json.loads((TRAINSET_DIR / 'manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest, ensure_ascii=False, indent=2))


In [ ]:
# Train QLoRA adapter for qwen3 and optionally push to Hub
import subprocess
from pathlib import Path

cmd = [
    'python', 'scripts/train_abo150_qlora.py',
    '--train-jsonl', str(TRAINSET_DIR / 'train.jsonl'),
    '--val-jsonl', str(TRAINSET_DIR / 'val.jsonl'),
    '--model-key', BASE_MODEL_KEY,
    '--output-dir', str(ADAPTER_DIR),
    '--num-train-epochs', str(NUM_TRAIN_EPOCHS),
    '--learning-rate', str(LEARNING_RATE),
    '--gradient-accumulation-steps', str(GRAD_ACC_STEPS),
    '--per-device-train-batch-size', str(TRAIN_BATCH_SIZE),
    '--per-device-eval-batch-size', str(EVAL_BATCH_SIZE),
    '--logging-steps', str(LOGGING_STEPS),
    '--save-steps', str(SAVE_STEPS),
    '--eval-steps', str(EVAL_STEPS),
]
if DISABLE_TQDM:
    cmd.append('--disable-tqdm')
if PUSH_TO_HUB:
    cmd.append('--push-to-hub')
if HUB_PRIVATE:
    cmd.append('--hub-private')
cmd.extend(['--comet-project-name', COMET_PROJECT_NAME])
if not COMET_ENABLED:
    cmd.append('--disable-comet')
if HUB_MODEL_ID:
    cmd.extend(['--hub-model-id', HUB_MODEL_ID])

log_path = ADAPTER_DIR / 'train_stdout.log'
log_path.parent.mkdir(parents=True, exist_ok=True)
print('>>', ' '.join(cmd))
print('Log path:', log_path)

with log_path.open('w', encoding='utf-8') as log_handle:
    process = subprocess.Popen(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    assert process.stdout is not None
    for line in process.stdout:
        print(line, end='')
        log_handle.write(line)
    return_code = process.wait()

if return_code != 0:
    raise RuntimeError(f'Training failed with exit code {return_code}. Check {log_path}')

manifest = json.loads((ADAPTER_DIR / 'manifest.json').read_text(encoding='utf-8'))
print(json.dumps(manifest, ensure_ascii=False, indent=2))
print('Local runtime config:', ADAPTER_DIR / 'runtime_model_config.json')
hub_cfg = ADAPTER_DIR / 'runtime_model_config.hub.json'
print('Hub runtime config exists:', hub_cfg.exists())
if hub_cfg.exists():
    print('Hub runtime config:', hub_cfg)


In [ ]:
# Validate the adapter on the fixed 50-object holdout
import subprocess

hub_cfg = ADAPTER_DIR / 'runtime_model_config.hub.json'
local_cfg = ADAPTER_DIR / 'runtime_model_config.json'
model_config_path = hub_cfg if hub_cfg.exists() else local_cfg

cmd = [
    'python', 'scripts/run_vlm_validation.py',
    '--dataset-name', DATASET_NAME,
    '--protocol-name', PROTOCOL_NAME,
    '--sample-ids-path', str(VAL_IDS_PATH),
    '--model-config-path', str(model_config_path),
    '--custom-model-key', CUSTOM_MODEL_KEY,
    '--variants', ','.join(VALIDATION_VARIANTS),
    '--save-raw-output',
]
print('>>', ' '.join(cmd))
subprocess.run(cmd, check=True)

summary_path = Path('reports_abo150_expanded') / PROTOCOL_NAME / CUSTOM_MODEL_KEY / VALIDATION_VARIANTS[0] / 'summary.json'
print('Summary path:', summary_path)
print(summary_path.read_text(encoding='utf-8'))


In [ ]:
# Optional: inspect the main result files
from pathlib import Path
import pandas as pd

run_dir = Path('reports_abo150_expanded') / PROTOCOL_NAME / CUSTOM_MODEL_KEY / VALIDATION_VARIANTS[0]
display(pd.read_csv(run_dir / 'property_metrics.csv').head(20))
display(pd.read_csv(run_dir / 'per_sample_predictions.csv').head(5))
